# VietMedBridge — 01: Crawl nguồn có checkpoint

Chạy sau notebook 00. Mặc định chỉ crawl mẫu domain nhỏ.
File raw chứa bytes HTTP đã giải nén content-encoding, hash, URL gốc,
URL sau redirect và trạng thái cho từng ID. Mỗi shard hoàn thành được lưu
vào Drive; khi Colab ngắt, chạy lại cell để tiếp tục từ checkpoint.

Dữ liệu có hàng triệu URL. Cần xem domain audit và chất lượng sample trước
khi chọn phạm vi lớn; các nguồn như PubMed/PMC có thể cần bulk/API adapter
để thu thập hiệu quả. Crawler HTTP ở đây là nền tảng khởi đầu.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "có thể chọn Runtime Version 2026.07 (Python 3.12)."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
drive.mount("/content/drive")

# Giữ cùng DATA_ROOT trong cả ba notebook.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
reference = CODE_REVISION or lock.get("git_commit") or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True
).strip()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(CHECKOUT) + "[notebook]"],
    check=True,
)
from vietmedbridge.artifacts import atomic_json, runtime_versions
if not lock or CODE_REVISION:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT})
atomic_json(DATA_ROOT / "runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 1. Chọn input và phạm vi theo vị trí dòng Parquet


In [ ]:
from vietmedbridge.dataset import load_snapshot, parquet_path, validate_link_subset
from vietmedbridge.crawl import CrawlConfig, crawl_links, completed_parts
from vietmedbridge.artifacts import read_json, verify_file

SNAPSHOT = load_snapshot(DATA_ROOT)
OFFICIAL_LINKS = parquet_path(DATA_ROOT, "links_corpus.parquet")
MODE = "smoke"       # "smoke" hoặc "range"
START_ROW = 0        # vị trí dòng, KHÔNG phải official doc_id
STOP_ROW = 5000     # chỉ dùng ở mode range; None = hết file
WORKER_INDEX = 0
WORKER_COUNT = 1     # workers phải có cùng range và xử lý shard khác nhau
MAX_NEW_SHARDS = 10  # số shard mới mỗi lần chạy; None = hết range
RETRY_FAILED = False

if MODE == "smoke":
    audit = read_json(DATA_ROOT / "reports/dataset_audit.json")
    if audit["corpus_sha256"] != SNAPSHOT["files"]["links_corpus.parquet"]["sha256"]:
        raise ValueError("Audit/sample không thuộc snapshot đang dùng; chạy lại notebook 00.")
    INPUT_LINKS = DATA_ROOT / audit["sample"]["path"]
    verify_file(INPUT_LINKS, audit["sample"]["sha256"])
    validate_link_subset(INPUT_LINKS, OFFICIAL_LINKS, work_dir=WORK_DIR)
    RUN_NAME = "smoke-v1"
    START, STOP = 0, None
elif MODE == "range":
    INPUT_LINKS = OFFICIAL_LINKS
    RUN_NAME = "corpus-v1"
    START, STOP = START_ROW, STOP_ROW
else:
    raise ValueError("MODE phải là smoke hoặc range.")

print("Run:", RUN_NAME, "| input:", INPUT_LINKS.name, "| row range:", START, STOP)


## 2. Crawl theo shard

Giữ cùng shard_size, range và cấu hình khi resume. Đổi range, cấu hình hoặc input phải
dùng RUN_NAME mới. Mỗi host có giới hạn tốc độ; 429/5xx được retry có backoff.
Robots bị chặn, lỗi tải, file quá lớn và các lỗi khác được lưu vào ledger.
Một shard ghi đủ trạng thái cho mọi ID mới được coi là hoàn thành.


In [ ]:
CONFIG = CrawlConfig(
    concurrency=4, shard_size=128, per_host_delay=1.0,
    timeout_seconds=30, max_bytes=16 * 1024 * 1024,
    attempts=3, respect_robots=True,
)
SUMMARY = await crawl_links(
    INPUT_LINKS, DATA_ROOT / "crawl", run_name=RUN_NAME, config=CONFIG,
    start=START, stop=STOP, worker_index=WORKER_INDEX, worker_count=WORKER_COUNT,
    retry_failed=RETRY_FAILED, max_shards=MAX_NEW_SHARDS, work_dir=WORK_DIR,
    origin_corpus_sha256=SNAPSHOT["files"]["links_corpus.parquet"]["sha256"],
)
print(json.dumps(SUMMARY, ensure_ascii=False, indent=2))


## 3. Kiểm tra coverage và các ID lỗi

Số recorded_documents bên dưới chỉ là phần đã crawl trong run.
Sample thành công không có nghĩa đã thu thập đủ corpus chính thức.


In [ ]:
import pandas as pd
PARTS = completed_parts(DATA_ROOT / "crawl" / RUN_NAME)
from itertools import islice
failed_count = sum(len(part["failed_ids"]) for part in PARTS)
failed_preview = list(islice(
    ({"part": part["part"], "doc_id": doc_id}
     for part in PARTS for doc_id in part["failed_ids"]), 100,
))
print("Official corpus rows:", SNAPSHOT["files"]["links_corpus.parquet"]["rows"])
print("Recorded:", SUMMARY["recorded_documents"], "| errors:", failed_count)
display(pd.DataFrame([{"status": key, "count": value} for key, value in SUMMARY["statuses"].items()]))
if failed_preview:
    display(pd.DataFrame(failed_preview))


## Resume và retry

Chạy lại cell 2 để xử lý các shard còn thiếu trong range. Đặt RETRY_FAILED=True
rồi chạy lại cell 1–2 để chỉ tải lại ID thất bại trong shard đã hoàn thành.
Các nguồn thành công được giữ lại, và raw attempt cũ vẫn tồn tại.
Không chạy hai runtime ghi cùng shard; dùng WORKER_INDEX/WORKER_COUNT để chia việc.
Sau sample, kiểm tra tỷ lệ lỗi theo domain trước khi mở rộng phạm vi.
